In [1]:
import pandas as pd

# ── 데이터 로드 ───────────────────────────────────────────────────────────
df = pd.read_csv('reviews_step2_dedup.csv', encoding='utf-8-sig', low_memory=False)
print(f"로드 완료: {len(df):,}건")
print(f"컬럼 목록: {df.columns.tolist()}")

# action 컬럼 확인
if 'action' in df.columns:
    print(f"\naction 분포:\n{df['action'].value_counts()}")

로드 완료: 623,665건
컬럼 목록: ['리뷰번호', 'goodsNo', '리뷰타입', '작성자', '리뷰내용', '평점', '체험단', '구매옵션', '키', '몸무게', '성별', '작성일', '만족도', '사진유무', '도움돼요', '구매사이즈', '구매상세', '연도', '월', '일', '요일', '평점_raw', '만족도_응답여부', '사이즈', '화면대비색감', '퀄리티', '구김', '두께감', '신축성', '색감', '보온성', '퀄리티_점수', '보온성_점수', '신축성_점수', '두께감_점수', '구김_점수', '사이즈_편차', '사이즈_편차절대', '화면대비색감_편차', '화면대비색감_편차절대', '색감_편차', '색감_편차절대', '노출일수', '일평균_도움돼요지수', '도움여부', '리뷰내용_clean', '리뷰내용_norm', 'laugh_count', 'cry_count', 'exclamation_count', 'question_count', 'emoji_count', 'has_repetition', 'text_length_orig', '한글_글자수', '전체_글자수', 'is_valid_for_topic', '작성자_norm', '옵션키', '세션', 'long_gap_review', '그룹크기', 'action', 'dup_flag', 'is_base', 'kept_id']

action 분포:
action
keep    623665
Name: count, dtype: int64


In [2]:
# ── keep된 리뷰 중 같은 그룹 안에서 리뷰내용_norm이 완전히 같은 케이스 확인 ──
dup_in_keep = df[
    df.duplicated(subset=['작성자_norm', 'goodsNo', '세션', '리뷰내용_norm'], keep=False)
]

print(f"전체 리뷰: {len(df):,}건")
print(f"keep 안에서 중복 텍스트 건수: {len(dup_in_keep):,}건")
print(f"영향받은 그룹 수: {dup_in_keep.groupby(['작성자_norm', 'goodsNo', '세션']).ngroups:,}그룹")

# ── 샘플 확인 ─────────────────────────────────────────────────────────────
print("\n=== 샘플 ===")
groups = list(dup_in_keep.groupby(['작성자_norm', 'goodsNo', '세션']))[:3]
for (author, goods, session), rows in groups:
    print(f"\n작성자={author} | 상품={goods} | 세션={session}")
    for _, row in rows.iterrows():
        print(f"  리뷰번호={row['리뷰번호']} | dup_flag={row['dup_flag']}")
        print(f"  텍스트: {str(row['리뷰내용_norm'])[:80]}")

전체 리뷰: 623,665건
keep 안에서 중복 텍스트 건수: 890건
영향받은 그룹 수: 434그룹

=== 샘플 ===

작성자='#^_^ | 상품=3791891 | 세션=0
  리뷰번호=67066326 | dup_flag=month_excluded
  텍스트: 세탁돌려도크게헤지는거없고나름따뜻해요핏은좀애매한느낌이있어요
  리뷰번호=67066340 | dup_flag=month_excluded
  텍스트: 세탁돌려도크게헤지는거없고나름따뜻해요핏은좀애매한느낌이있어요

작성자='@#?!@ | 상품=2786732 | 세션=0
  리뷰번호=70270844 | dup_flag=unique
  텍스트: 너무이쁘고짱짱편해요자주입고다닐거같아요
  리뷰번호=70270795 | dup_flag=month_excluded
  텍스트: 너무이쁘고짱짱편해요자주입고다닐거같아요

작성자=00013130 | 상품=3616463 | 세션=0
  리뷰번호=71128213 | dup_flag=month_excluded
  텍스트: 좋습니다편하게입기좋아요또구매할의향있습니다
  리뷰번호=71128220 | dup_flag=month_excluded
  텍스트: 좋습니다편하게입기좋아요또구매할의향있습니다


In [3]:
# month_excluded 제외하고 확인
dup_in_keep_no_month = dup_in_keep[
    ~dup_in_keep['dup_flag'].str.contains('month_excluded')
]

# 같은 그룹 안에서 month_excluded가 아닌 것끼리만 중복인 경우
dup_no_month = df[df['dup_flag'] != 'month_excluded'].copy()
dup_no_month = dup_no_month[
    dup_no_month.duplicated(subset=['작성자_norm', 'goodsNo', '세션', '리뷰내용_norm'], keep=False)
]

print(f"month 제외 후 중복 텍스트 건수: {len(dup_no_month):,}건")
print(f"영향받은 그룹 수: {dup_no_month.groupby(['작성자_norm', 'goodsNo', '세션']).ngroups:,}그룹")

# dup_flag 분포 확인
print(f"\ndup_flag 분포:\n{dup_no_month['dup_flag'].value_counts()}")

month 제외 후 중복 텍스트 건수: 5건
영향받은 그룹 수: 2그룹

dup_flag 분포:
dup_flag
unique    5
Name: count, dtype: int64


In [4]:
# unique 제외, month 제외한 중복 케이스 상세 확인
dup_real = dup_no_month[dup_no_month['dup_flag'] != 'unique']

print(f"건수: {len(dup_real):,}건")
print(f"그룹 수: {dup_real.groupby(['작성자_norm', 'goodsNo', '세션']).ngroups:,}그룹")

print("\n=== 상세 내용 ===")
for (author, goods, session), rows in dup_real.groupby(['작성자_norm', 'goodsNo', '세션']):
    print(f"\n작성자={author} | 상품={goods} | 세션={session}")
    for _, row in rows.iterrows():
        print(f"  리뷰번호={row['리뷰번호']} | dup_flag={row['dup_flag']} | 타입={row['리뷰타입']} | 옵션={row['옵션키']}")
        print(f"  텍스트: {str(row['리뷰내용_norm'])[:100]}")

건수: 0건
그룹 수: 0그룹

=== 상세 내용 ===


In [5]:
rows = df[df['리뷰번호'].isin([9798879, 9799070])]
for _, row in rows.iterrows():
    print(f"리뷰번호={row['리뷰번호']}")
    print(f"리뷰내용_norm='{row['리뷰내용_norm']}'")
    print(f"타입={row['리뷰타입']} | 옵션={row['옵션키']}")
    print()

In [7]:
# 같은 그룹 전체 확인
group = df[
    (df['작성자_norm'] == '아니패션이뭔데') &
    (df['goodsNo'] == 1291015) &
    (df['세션'] == 0)
][['리뷰번호', '리뷰타입', '옵션키', 'dup_flag', 'action', 'is_base', 'kept_id', '리뷰내용_norm']]

print(f"그룹 크기: {len(group)}개")
print()
for _, row in group.iterrows():
    print(f"리뷰번호={row['리뷰번호']} | 타입={row['리뷰타입']} | 옵션={row['옵션키']}")
    print(f"  dup_flag={row['dup_flag']} | is_base={row['is_base']} | kept_id={row['kept_id']}")
    print(f"  텍스트: {str(row['리뷰내용_norm'])[:60]}")
    print()

그룹 크기: 2개

리뷰번호=9798940 | 타입=style | 옵션=('2XL', '아보카도')
  dup_flag=multi_both_dup | is_base=True | kept_id=9798940
  텍스트: 흔하지않아서좋고색이산뜻해서봄에도잘입었습니당엄청많이커요

리뷰번호=9799065 | 타입=photo | 옵션=('2XL', '아보카도')
  dup_flag=multi_type_unique | is_base=False | kept_id=9798940
  텍스트: 흔하지않아서좋고색이산뜻해서봄에도잘입었습니당크고얇아요

